# einops-repeat — worked example 2: Tile a sequence tensor twice along the time axis

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-repeat`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import torch.nn.functional as F
import einops
from einops import repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The `(r t)` composition on the output side of `repeat` tiles the sequence: the full original sequence appears `r` times consecutively. The outer factor `r` varies slowest, so the first copy of the sequence comes first, then the second copy. This is the `repeat` pattern for duplication rather than stretching.

## Worked solution

Input: sequence of shape `(B=2, T=5, D=8)`.

**Pattern:** `'b t d -> b (r t) d'` with `r=2`.

The `(r t)` group on the output puts `r` as the outer index (varies slower) and `t` as the inner (varies faster). This means positions `0..T-1` of the output are copy 1 of the input, and positions `T..2T-1` are copy 2.

**Result shape:** `(2, 10, 8)`. Each sequence is repeated twice: `[s0, s1, s2, s3, s4, s0, s1, s2, s3, s4]`.

In [ ]:
import torch as t
from einops import repeat

t.manual_seed(25)
B, T, D = 2, 5, 8
x = t.randn(B, T, D)

def tile_sequence_twice(x):
    return repeat(x, 'b t d -> b (r t) d', r=2)

tiled = tile_sequence_twice(x)
print('Input shape:', x.shape)   # (2, 5, 8)
print('Tiled shape:', tiled.shape)  # (2, 10, 8)
assert tiled.shape == (B, 2 * T, D)

# First half equals original, second half also equals original
assert t.allclose(tiled[:, :T, :], x)
assert t.allclose(tiled[:, T:, :], x)
print('First and second halves match original:', True)